# Exoplanet KOI Classification — Improved

## What this notebook does

This notebook classifies **Kepler Objects of Interest (KOIs)** as:

- `CONFIRMED` → `1`
- `FALSE POSITIVE` → `0`

It compares a majority-class baseline, Logistic Regression, and a Decision Tree. The improved workflow uses **host-star-aware train/test splitting and cross-validation**, removes exact feature duplication, keeps preprocessing inside pipelines, and tunes the signal-to-noise (SNR) ablation fairly.

> **Scientific scope:** this is catalog-level classification of objects that have already been identified as KOIs. It is **not yet a system that discovers transits from raw brightness-versus-time light curves**. Catalog fields such as transit depth and model SNR are derived from transit analyses and may overlap conceptually with the evidence used by the catalog vetting process. Results must be described as reproducing catalog dispositions, not as independently discovering new planets.

### Improvements in this version

1. Keep all KOIs from the same host star (`kepid`) together in either train or test data.
2. Use `StratifiedGroupKFold` for model-selection cross-validation too.
3. Replace raw period/depth/SNR with one log-transformed version of each; avoid storing both depth and the same depth in fractional/percentage units.
4. Keep invalid/missing feature values for pipeline imputation instead of dropping otherwise usable rows.
5. Tune the full-SNR and no-SNR Logistic Regression models with the same grid and group-aware folds.
6. Report class-aware metrics and make the scientific limitation explicit.

## 1. Imports and reproducibility

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score, classification_report,
    ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay
)
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)
pd.set_option("display.max_columns", 100)

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

## 2. Load the supplied dataset

In [ ]:
# Works when cumulative.csv is beside this notebook, or in /mnt/data in hosted environments.
possible_paths = [Path("cumulative.csv"), Path("/mnt/data/cumulative.csv")]
DATA_PATH = next((p for p in possible_paths if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find cumulative.csv. Put it in the same folder as this notebook "
        "or update DATA_PATH to its location."
    )

df = pd.read_csv(DATA_PATH, low_memory=False)
print(f"Loaded: {DATA_PATH.resolve()}")
print(f"Rows: {len(df):,} | Columns: {df.shape[1]}")
display(df.head())

## 3. Inspect labels, missing values, and repeated host stars

In [ ]:
required_columns = [
    "koi_disposition", "kepid", "koi_period", "koi_duration", "koi_depth",
    "koi_model_snr", "koi_steff", "koi_slogg", "koi_srad"
]
missing_columns = [c for c in required_columns if c not in df.columns]
if missing_columns:
    raise KeyError(f"The dataset is missing required columns: {missing_columns}")

print("Disposition counts in source data:")
display(df["koi_disposition"].value_counts(dropna=False).rename("count").to_frame())
print("\nTop missing-value counts:")
display(df.isna().sum().sort_values(ascending=False).head(15).rename("missing_count").to_frame())
print("\nExact duplicate rows:", int(df.duplicated().sum()))
print("Unique host stars (kepid):", int(df["kepid"].nunique()))
host_counts = df["kepid"].value_counts()
print("Host stars with multiple KOI rows:", int((host_counts > 1).sum()))
print("KOI rows whose host has multiple rows:", int(host_counts[host_counts > 1].sum()))

## 4. Define the binary classification task

We exclude `CANDIDATE` rows from this beginner/intermediate binary task rather than treating them as negatives. Candidate objects are an unresolved class, not equivalent to false positives.

The target is an existing catalog disposition. This means a successful model is learning patterns associated with catalog labels; it is not proof that it can independently confirm an exoplanet.

In [ ]:
TARGET_MAP = {"CONFIRMED": 1, "FALSE POSITIVE": 0}
df_model = df.loc[df["koi_disposition"].isin(TARGET_MAP)].copy()
df_model["LABEL"] = df_model["koi_disposition"].map(TARGET_MAP).astype(int)

# A host-star identifier is required for leakage-resistant grouped evaluation.
if df_model["kepid"].isna().any():
    raise ValueError("Some selected rows have no kepid; resolve these before grouped evaluation.")

df_model["HOST_ID"] = df_model["kepid"].astype(str)
print("Rows after keeping CONFIRMED / FALSE POSITIVE:", f"{len(df_model):,}")
print("Unique host stars in selected sample:", f"{df_model['HOST_ID'].nunique():,}")
display(pd.crosstab(df_model["koi_disposition"], df_model["LABEL"]))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
counts = df_model["koi_disposition"].value_counts().reindex(["FALSE POSITIVE", "CONFIRMED"])
counts.plot(kind="bar", ax=ax)
ax.set_title("Target distribution among selected KOIs")
ax.set_ylabel("Number of KOIs")
ax.set_xlabel("")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print("Positive-class prevalence (CONFIRMED): {:.1%}".format(df_model["LABEL"].mean()))

## 5. Feature design: one representation per measurement

We use eight model features:

| Model feature | Source / construction | Interpretation |
|---|---|---|
| `log_period` | `log1p(koi_period)` | Log orbital period (days) |
| `koi_duration` | `koi_duration` | Transit duration (hours) |
| `log_depth` | `log1p(koi_depth)` | Log transit depth (ppm) |
| `log_snr` | `log1p(koi_model_snr)` | Log transit-model signal-to-noise ratio |
| `koi_steff` | `koi_steff` | Stellar effective temperature (K) |
| `koi_slogg` | `koi_slogg` | Stellar surface gravity (log cgs) |
| `koi_srad` | `koi_srad` | Stellar radius (solar radii) |
| `transit_duty_cycle` | duration / (24 × period) | Fraction of an orbit spent in transit |

The main model does **not** include `koi_depth`, `transit_depth_fraction`, and `transit_depth_percent` simultaneously, because these are the same depth measurement on different scales. Likewise, `koi_period` and `koi_model_snr` are represented by their log transforms rather than both raw and log versions.

Non-finite or physically invalid entries are converted to missing values and imputed **inside** each model pipeline. Duty-cycle ratios above 1 are treated as invalid because a transit cannot physically last longer than a full orbital period; such values can occur in poorly fitted false-positive signals. This avoids discarding complete rows solely because one feature is missing.

In [ ]:
RAW_NUMERIC = [
    "koi_period", "koi_duration", "koi_depth", "koi_model_snr",
    "koi_steff", "koi_slogg", "koi_srad"
]
for col in RAW_NUMERIC:
    df_model[col] = pd.to_numeric(df_model[col], errors="coerce")
    df_model.loc[~np.isfinite(df_model[col]), col] = np.nan

# Log transforms require non-negative values. Invalid values become NaN and will be imputed.
for col in ["koi_period", "koi_duration", "koi_depth", "koi_model_snr"]:
    df_model.loc[df_model[col] < 0, col] = np.nan

# Period and duration must be positive for the derived duty-cycle calculation.
df_model.loc[df_model["koi_period"] <= 0, "koi_period"] = np.nan
df_model.loc[df_model["koi_duration"] <= 0, "koi_duration"] = np.nan

# Stellar temperature/radius should be positive; surface gravity is left as reported.
for col in ["koi_steff", "koi_srad"]:
    df_model.loc[df_model[col] <= 0, col] = np.nan

df_model["log_period"] = np.log1p(df_model["koi_period"])
df_model["log_depth"] = np.log1p(df_model["koi_depth"])
df_model["log_snr"] = np.log1p(df_model["koi_model_snr"])
df_model["transit_duty_cycle"] = df_model["koi_duration"] / (24 * df_model["koi_period"])
df_model.loc[~np.isfinite(df_model["transit_duty_cycle"]), "transit_duty_cycle"] = np.nan
# A transit duration cannot physically exceed an entire orbital period. Treat such fit-derived ratios as invalid.
invalid_duty_cycle = df_model["transit_duty_cycle"] > 1
print("Physically implausible duty-cycle values (> 1) set to missing:", int(invalid_duty_cycle.sum()))
df_model.loc[invalid_duty_cycle, "transit_duty_cycle"] = np.nan

FEATURES = [
    "log_period", "koi_duration", "log_depth", "log_snr",
    "koi_steff", "koi_slogg", "koi_srad", "transit_duty_cycle"
]

X = df_model[FEATURES].copy()
y = df_model["LABEL"].copy()
groups = df_model["HOST_ID"].copy()

print("Feature matrix:", X.shape)
print("Features:", FEATURES)
print("Missing values per model feature (imputed within CV pipelines):")
display(X.isna().sum().sort_values(ascending=False).rename("missing_count").to_frame())
display(X.describe().T)

## 6. Exploratory checks

These plots describe the input data; they are not used to decide hyperparameters using the test set.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(data=df_model, x="log_snr", hue="koi_disposition", bins=35, element="step", stat="density", common_norm=False, ax=axes[0])
axes[0].set_title("Log-transformed transit-model SNR")
sns.histplot(data=df_model, x="log_depth", hue="koi_disposition", bins=35, element="step", stat="density", common_norm=False, ax=axes[1])
axes[1].set_title("Log-transformed transit depth")
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 6))
sns.heatmap(X.corr(), annot=True, fmt=".2f", center=0, cmap="vlag")
plt.title("Correlation between model features")
plt.tight_layout()
plt.show()

## 7. Group-aware train/test split

A random row-wise split can put one KOI from a star in training and another KOI from the same star in testing. Models might then benefit from star-specific similarities. To reduce this risk, we use `StratifiedGroupKFold`: `kepid` values never cross the train/test boundary, while the fold assignment attempts to preserve class proportions.

One fold is reserved as the final test set. The remaining data are used for model selection with a **separate group-aware cross-validation procedure**.

In [ ]:
outer_splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
train_idx, test_idx = next(outer_splitter.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
y_train, y_test = y.iloc[train_idx].copy(), y.iloc[test_idx].copy()
groups_train, groups_test = groups.iloc[train_idx].copy(), groups.iloc[test_idx].copy()

train_hosts = set(groups_train)
test_hosts = set(groups_test)
overlap = train_hosts.intersection(test_hosts)
assert len(overlap) == 0, f"Host-star leakage detected: {len(overlap)} shared host IDs"

print(f"Train rows: {len(X_train):,} ({len(X_train)/len(X):.1%})")
print(f"Test rows:  {len(X_test):,} ({len(X_test)/len(X):.1%})")
print(f"Train unique hosts: {groups_train.nunique():,}")
print(f"Test unique hosts:  {groups_test.nunique():,}")
print("Overlapping hosts:", len(overlap))

split_summary = pd.DataFrame({
    "Train": [len(y_train), int(y_train.sum()), y_train.mean()],
    "Test": [len(y_test), int(y_test.sum()), y_test.mean()]
}, index=["Rows", "Confirmed rows", "Confirmed prevalence"])
display(split_summary)

## 8. Baseline model

A majority-class classifier shows how misleading accuracy can be when one label is more common. We report balanced accuracy and positive-class metrics too.

In [ ]:
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)
dummy_pred = dummy.predict(X_test)

dummy_metrics = {
    "Model": "Majority-class baseline",
    "Accuracy": accuracy_score(y_test, dummy_pred),
    "Balanced accuracy": balanced_accuracy_score(y_test, dummy_pred),
    "Precision": precision_score(y_test, dummy_pred, zero_division=0),
    "Recall": recall_score(y_test, dummy_pred, zero_division=0),
    "F1": f1_score(y_test, dummy_pred, zero_division=0),
    "ROC-AUC": np.nan,
    "Average precision": np.nan,
}
display(pd.DataFrame([dummy_metrics]).set_index("Model"))

## 9. Pipelines and group-aware cross-validation

- The Logistic Regression pipeline median-imputes missing values, standardizes features, then fits the classifier.
- The Decision Tree pipeline median-imputes values but does not scale features; tree split rules do not require standardization.
- Imputation and scaling are fitted separately inside each fold, preventing information from a validation fold from influencing preprocessing.
- Hyperparameters are chosen using training data only. The final test set is kept out of model selection.

In [ ]:
def make_logistic_pipeline():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=3000, random_state=RANDOM_STATE))
    ])

def make_tree_pipeline():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", DecisionTreeClassifier(random_state=RANDOM_STATE))
    ])

INNER_SPLITS = 5
cv = StratifiedGroupKFold(n_splits=INNER_SPLITS, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "average_precision": "average_precision",
}

# Confirm folds stay group-separated and contain both classes.
for fold, (tr_i, va_i) in enumerate(cv.split(X_train, y_train, groups_train), start=1):
    shared = set(groups_train.iloc[tr_i]).intersection(set(groups_train.iloc[va_i]))
    assert not shared, f"Group overlap in inner CV fold {fold}"
    assert y_train.iloc[tr_i].nunique() == 2 and y_train.iloc[va_i].nunique() == 2
print("Verified: all inner CV folds keep host stars separated and contain both classes.")

## 10. Tune Logistic Regression

F1 is the main grid-search selection metric because the goal is to balance precision and recall for the `CONFIRMED` class. We still report a broader set of metrics after training.

In [ ]:
logistic_pipe = make_logistic_pipeline()
LOGISTIC_GRID = {
    "model__C": [0.1, 1.0, 10.0],
    "model__class_weight": [None, "balanced"],
}

log_search = GridSearchCV(
    estimator=logistic_pipe,
    param_grid=LOGISTIC_GRID,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    refit=True,
    return_train_score=True,
    error_score="raise",
)
log_search.fit(X_train, y_train, groups=groups_train)
print("Best Logistic Regression parameters:", log_search.best_params_)
print("Best mean grouped-CV F1: {:.4f}".format(log_search.best_score_))

## 11. Tune Decision Tree

The search below is intentionally compact enough to run on a laptop while testing regularization choices that limit overfitting.

In [ ]:
tree_pipe = make_tree_pipeline()
TREE_GRID = {
    "model__criterion": ["gini", "entropy"],
    "model__max_depth": [3, 5, 8, None],
    "model__min_samples_leaf": [2, 5, 10],
    "model__class_weight": [None, "balanced"],
}

tree_search = GridSearchCV(
    estimator=tree_pipe,
    param_grid=TREE_GRID,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    refit=True,
    return_train_score=True,
    error_score="raise",
)
tree_search.fit(X_train, y_train, groups=groups_train)
print("Best Decision Tree parameters:", tree_search.best_params_)
print("Best mean grouped-CV F1: {:.4f}".format(tree_search.best_score_))

## 12. Final held-out test evaluation

The test metrics estimate performance on host stars that were not used for training or tuning. The positive class is `CONFIRMED` (`1`). Average precision summarizes the precision-recall ranking curve; it is reported explicitly rather than implying that it is a simple threshold-specific metric.

In [ ]:
models = {
    "Majority-class baseline": dummy,
    "Logistic Regression": log_search.best_estimator_,
    "Decision Tree": tree_search.best_estimator_,
}

def evaluate_classifier(name, model, X_eval, y_eval):
    pred = model.predict(X_eval)
    row = {
        "Model": name,
        "Accuracy": accuracy_score(y_eval, pred),
        "Balanced accuracy": balanced_accuracy_score(y_eval, pred),
        "Precision": precision_score(y_eval, pred, zero_division=0),
        "Recall": recall_score(y_eval, pred, zero_division=0),
        "F1": f1_score(y_eval, pred, zero_division=0),
    }
    if hasattr(model, "predict_proba") and y_eval.nunique() == 2:
        prob = model.predict_proba(X_eval)[:, 1]
        row["ROC-AUC"] = roc_auc_score(y_eval, prob)
        row["Average precision"] = average_precision_score(y_eval, prob)
    else:
        row["ROC-AUC"] = np.nan
        row["Average precision"] = np.nan
    return row

results_df = pd.DataFrame([
    evaluate_classifier(name, model, X_test, y_test)
    for name, model in models.items()
]).sort_values("F1", ascending=False).reset_index(drop=True)

display(results_df.style.format({c: "{:.3f}" for c in results_df.columns if c != "Model"}))

### Confusion matrices and classification reports

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, name in zip(axes, ["Logistic Regression", "Decision Tree"]):
    ConfusionMatrixDisplay.from_estimator(
        models[name], X_test, y_test,
        display_labels=["False positive", "Confirmed"],
        values_format="d", ax=ax, colorbar=False
    )
    ax.set_title(name)
plt.tight_layout()
plt.show()

for name in ["Logistic Regression", "Decision Tree"]:
    print("\n", name)
    print(classification_report(
        y_test, models[name].predict(X_test),
        labels=[0, 1], target_names=["False positive", "Confirmed"],
        zero_division=0, digits=3
    ))

### ROC and precision-recall curves

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for name in ["Logistic Regression", "Decision Tree"]:
    RocCurveDisplay.from_estimator(models[name], X_test, y_test, name=name, ax=ax)
ax.set_title("ROC curves — grouped held-out test set")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(7, 5))
for name in ["Logistic Regression", "Decision Tree"]:
    PrecisionRecallDisplay.from_estimator(models[name], X_test, y_test, name=name, ax=ax)
ax.set_title("Precision-recall curves — grouped held-out test set")
plt.tight_layout()
plt.show()

## 13. Interpret model features carefully

These are descriptive associations, not causal effects. Because the inputs include quantities derived from the transit fit, a feature ranking does not independently establish which physical process caused the catalog disposition.

In [ ]:
best_logistic = log_search.best_estimator_
log_model = best_logistic.named_steps["model"]
coef_table = pd.DataFrame({
    "Feature": FEATURES,
    "Standardized coefficient": log_model.coef_[0],
    "Absolute coefficient": np.abs(log_model.coef_[0]),
}).sort_values("Absolute coefficient", ascending=False)
display(coef_table)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(coef_table["Feature"], coef_table["Standardized coefficient"])
ax.invert_yaxis()
ax.set_title("Logistic Regression coefficients")
plt.tight_layout()
plt.show()

best_tree = tree_search.best_estimator_
tree_model = best_tree.named_steps["model"]
tree_importance = pd.DataFrame({
    "Feature": FEATURES,
    "Impurity importance": tree_model.feature_importances_,
}).sort_values("Impurity importance", ascending=False)
display(tree_importance)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(tree_importance["Feature"], tree_importance["Impurity importance"])
ax.invert_yaxis()
ax.set_title("Decision Tree impurity-based feature importance")
plt.tight_layout()
plt.show()

### Permutation importance (descriptive diagnostic)

Permutation importance measures the drop in a chosen metric when a feature is shuffled. Use it to understand the already-selected model, **not** to repeatedly redesign the model based on the held-out test set.

In [ ]:
perm = permutation_importance(
    best_logistic,
    X_test,
    y_test,
    scoring="f1",
    n_repeats=10,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
perm_table = pd.DataFrame({
    "Feature": FEATURES,
    "Mean F1 decrease": perm.importances_mean,
    "Std": perm.importances_std,
}).sort_values("Mean F1 decrease", ascending=False)
display(perm_table)

## 14. Error analysis

We summarize false positives and false negatives. A false negative here means a catalog-confirmed KOI predicted as a false positive by our classifier; it does **not** mean an undiscovered planet was missed in raw telescope data.

In [ ]:
error_frame = X_test.copy()
error_frame["actual_label"] = y_test
error_frame["actual_disposition"] = np.where(y_test.to_numpy() == 1, "CONFIRMED", "FALSE POSITIVE")

for name in ["Logistic Regression", "Decision Tree"]:
    pred = models[name].predict(X_test)
    fp_mask = (y_test.to_numpy() == 0) & (pred == 1)
    fn_mask = (y_test.to_numpy() == 1) & (pred == 0)
    print(f"\n{name}")
    print("False positives:", int(fp_mask.sum()))
    print("False negatives:", int(fn_mask.sum()))
    if fp_mask.any():
        print("Median input features for false positives:")
        display(X_test.iloc[np.flatnonzero(fp_mask)].median().rename("median").to_frame())
    if fn_mask.any():
        print("Median input features for false negatives:")
        display(X_test.iloc[np.flatnonzero(fn_mask)].median().rename("median").to_frame())

## 15. Fair SNR ablation experiment

We compare two otherwise equivalent Logistic Regression searches:

- **With SNR:** all features listed in `FEATURES`.
- **Without SNR:** remove `log_snr`.

Both searches use the same hyperparameter grid, grouped cross-validation splits, scoring metric, and test set. This makes the comparison much fairer than comparing a tuned full model to an untuned ablation model.

SNR is a processed signal-quality measure, so a performance drop indicates model dependence on this catalog signal—not proof of independent planet discovery.

In [ ]:
NO_SNR_FEATURES = [f for f in FEATURES if f != "log_snr"]
X_train_no_snr = X_train[NO_SNR_FEATURES]
X_test_no_snr = X_test[NO_SNR_FEATURES]

no_snr_search = GridSearchCV(
    estimator=make_logistic_pipeline(),
    param_grid=LOGISTIC_GRID,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    refit=True,
    return_train_score=True,
    error_score="raise",
)
no_snr_search.fit(X_train_no_snr, y_train, groups=groups_train)

ablation_models = {
    "With SNR": log_search.best_estimator_,
    "Without SNR": no_snr_search.best_estimator_,
}

def evaluate_ablation(name, model, X_eval, y_eval):
    pred = model.predict(X_eval)
    prob = model.predict_proba(X_eval)[:, 1]
    return {
        "Experiment": name,
        "Best CV F1": model.best_score_ if hasattr(model, "best_score_") else np.nan,
        "Test precision": precision_score(y_eval, pred, zero_division=0),
        "Test recall": recall_score(y_eval, pred, zero_division=0),
        "Test F1": f1_score(y_eval, pred, zero_division=0),
        "Test ROC-AUC": roc_auc_score(y_eval, prob),
        "Test average precision": average_precision_score(y_eval, prob),
    }

ablation_results = []
for name, search in [("With SNR", log_search), ("Without SNR", no_snr_search)]:
    features_for_run = X_test if name == "With SNR" else X_test_no_snr
    ablation_results.append({
        "Experiment": name,
        "Best mean grouped-CV F1": search.best_score_,
        "Test precision": precision_score(y_test, search.best_estimator_.predict(features_for_run), zero_division=0),
        "Test recall": recall_score(y_test, search.best_estimator_.predict(features_for_run), zero_division=0),
        "Test F1": f1_score(y_test, search.best_estimator_.predict(features_for_run), zero_division=0),
        "Test ROC-AUC": roc_auc_score(y_test, search.best_estimator_.predict_proba(features_for_run)[:, 1]),
        "Test average precision": average_precision_score(y_test, search.best_estimator_.predict_proba(features_for_run)[:, 1]),
    })

display(pd.DataFrame(ablation_results).set_index("Experiment").style.format("{:.3f}"))
print("With-SNR best parameters:", log_search.best_params_)
print("No-SNR best parameters:", no_snr_search.best_params_)

## 16. Visualize the tuned shallow portion of the tree

In [ ]:
plt.figure(figsize=(18, 8))
plot_tree(
    tree_model,
    feature_names=FEATURES,
    class_names=["False Positive", "Confirmed"],
    filled=True,
    rounded=True,
    max_depth=3,
    fontsize=8,
)
plt.title("Decision Tree — first three levels")
plt.tight_layout()
plt.show()

## 17. Final interpretation and limitations

Use the model results printed above to fill in the project report. Do not claim results that are not supported by the evaluation outputs.

### Checklist for scientific reporting

- **Task:** binary classification of `CONFIRMED` versus `FALSE POSITIVE` KOIs; candidate rows were excluded.
- **Split:** held-out test data are grouped by `kepid`; assert verified that there is no host-star overlap.
- **Validation:** model tuning used `StratifiedGroupKFold`, with imputation/scaling fit inside each pipeline fold.
- **Metrics:** report F1, recall, precision, balanced accuracy, ROC-AUC, and average precision. Compare with the majority-class baseline.
- **SNR ablation:** report the matched with-SNR and no-SNR experiments; do not attribute the entire score difference to astrophysical discovery ability.
- **Interpretation:** coefficients and importance values are associations, not causal effects.

### Key limitation

The current model predicts a **catalog disposition for known KOIs using derived catalog-level features**. A stronger follow-on project would process raw Kepler light curves (time and flux), detrend and normalize them carefully, search for periodic transits (for example with Box Least Squares), phase-fold candidate signals, and then validate candidate signals with an ML model. That next stage should use data splits that also avoid leakage between target stars and, where possible, separate observation campaigns or instruments.

## 18. Reproducibility notes

- Random state: `42`.
- Dataset: NASA Kepler cumulative KOI CSV supplied with the project (`cumulative.csv`). Record its exact download date/release in the report, since catalog dispositions and columns can change across releases.
- **Population caveat:** a cumulative KOI catalog is a vetted/compiled catalog, not a uniformly selected population for all possible stars and planets. Treat the scores as performance on this supplied sample; do not infer survey-wide planet occurrence rates from them. For population-level conclusions, choose and document a uniform catalog/data release.
- This notebook intentionally does not invent numerical claims in advance; its metrics are generated by running the code against the local CSV.
- If your installed scikit-learn version is very old and does not include `StratifiedGroupKFold`, upgrade scikit-learn before running this notebook.